# 04 · Score scale: calibration across cell types and recall-based thresholds

CisLink scores are probabilities on the scale of the K562 CRISPR training pairs. The calibration slope (logistic regression of the label on logit(score); 1 = scale preserved) measures whether that scale transfers to other cell types; the recall-based thresholds give a scale-free way to read scores.

In [1]:
import pandas as pd

from cislink import datasets, load_model
from cislink import evaluate as ev

P = {c: datasets.crispr_pairs(c) for c in ("K562", "HCT116", "GM12878")}   # cells with both models (WTC11: V2G only)
rows = []
for c, p in P.items():
    for m in ("CisLink-E2G", "CisLink-V2G"):
        r = ev.calibration(p.label, p[m]); r.update(set=c, model=m); rows.append(r)
pd.DataFrame(rows)[["set", "model", "slope", "intercept", "mean_score", "positive_rate"]].round(3)

,set,model,slope,intercept,mean_score,positive_rate
0,K562,CisLink-E2G,0.860,-0.023,0.055,0.061
1,K562,CisLink-V2G,0.841,-0.068,0.054,0.061
2,HCT116,CisLink-E2G,1.139,-0.758,0.129,0.091
3,HCT116,CisLink-V2G,1.062,-0.097,0.102,0.091
4,GM12878,CisLink-E2G,1.132,0.882,0.187,0.235
5,GM12878,CisLink-V2G,1.324,0.643,0.201,0.235


In [2]:
k = P["K562"]
for m in ("e2g", "v2g"):
    model = load_model(m)
    print(model.name, {r: round(ev.recall_threshold(k.label, k[model.name], r / 100), 3) for r in (70, 50)}, "released:", model.thresholds)

CisLink-E2G {70: 0.18, 50: 0.529} released: {'recall_70': 0.18, 'recall_50': 0.529, 'note': 'score at which held-out K562 CRISPR pairs reach the given recall'}
CisLink-V2G {70: 0.139, 50: 0.41} released: {'recall_70': 0.139, 'recall_50': 0.41, 'note': 'score at which held-out K562 CRISPR pairs reach the given recall'}


In [3]:
t = []
for c, p in P.items():
    for m in ("e2g", "v2g"):
        model = load_model(m); hit = p[model.name] >= model.threshold(70)
        t.append({"set": c, "model": model.name, "threshold": model.threshold(70), "recall": p.label[hit].sum() / p.label.sum(),
                  "precision": p.label[hit].mean(), "positive rate": p.label.mean()})
pd.DataFrame(t).round(3)

,set,model,threshold,recall,precision,positive rate
0,K562,CisLink-E2G,0.180,0.696,0.530,0.061
1,K562,CisLink-V2G,0.139,0.696,0.440,0.061
2,HCT116,CisLink-E2G,0.180,0.971,0.452,0.091
3,HCT116,CisLink-V2G,0.139,0.824,0.350,0.091
4,GM12878,CisLink-E2G,0.180,0.812,0.722,0.235
5,GM12878,CisLink-V2G,0.139,0.938,0.652,0.235


In the manuscript the scale transfers to HCT116 and GM12878 (E2G slopes 1.14 and 1.13) but scores are overconfident in osteoblasts (0.54–0.62); the 70%-recall threshold keeps 53–97% recall across cell types.